# Selected metrics v2 — tables and visual comparisons

Read-only watchboard. **Nothing here re-scores metrics**: tables read the saved `results.csv` files produced by `run_metric(_fast).py` and `run_voxel_metric_v2(_fast).py`; the figures read the saved event streams in `data_test_evt` and the GT in the dataset folder.

**How to use**
1. Edit the settings in the next cell (dataset, sequences, metric list, voxel bin, time windows, models).
2. *Run All*. Every figure is saved in `reports/visualize_selected_metric_v2/` (a `YYYYmmdd_HHMMSS` sub-folder is created when that folder already exists).

| Section | What it shows | Interactive |
|---|---|---|
| 1 | Benchmark table (rows = baselines + student_v7) | – |
| 2 | Our models table (every teacher/student checkpoint found) | – |
| 3 | Event image + x-t + y-t slices, one row per model (GT first) | overlay version: yes |
| 4 | Event rate vs time (same chart as `watch_voxel_metrics_v2`) | yes |
| 5 | Voxel comparison: signed voxels, error maps, error over time | error-over-time: yes |
| 6 | Event frames on white background, sequence picker | widgets |
| 7 | 3D event cloud: GT + selected models (ON / OFF / both) | yes |

Colors everywhere: **ON = red, OFF = blue**, white background. Best four rank levels in tables are blue (darkest = best).

In [ ]:
# ======================= SETTINGS — edit, then Run All =======================
from pathlib import Path

# TABLES: one separate table per dataset, averaged over that dataset's test sequences.
#   'all' -> every dataset folder found in data_test_evt/ ; or a list, e.g. ['bsergb', 'eds', 'hqevfi']
DATASETS = 'all'
VIS_DATASET = 'bsergb'      # dataset used by the FIGURES (sections 3–7)
SPLIT = 'test'

# Sequences used by the TABLES (normally 'all' = the whole test split of each dataset).
#   'all'              -> mean over every test sequence that has saved scores
#   'seq_name'         -> one sequence
#   ['seq_a', 'seq_b'] -> mean over these sequences
SEQUENCES = 'all'
MATCH_SEQUENCES = True      # True: inside each column, every row is averaged over the SAME sequences (fair comparison)

# Columns of both tables (order = column order). Accepted names (case-insensitive):
#   chamfer_full, event_count, event_ratio, on_percentage, isi_distribution, iei_distribution,
#   fano_factor, voxel_mae, voxel_rmse, voxel_relative_l1, pmse_2_v2ce_adapted, pmse_4_v2ce_adapted
TABLE_METRICS = ['chamfer_full', 'event_count', 'event_ratio', 'on_percentage',
                 'isi_distribution', 'iei_distribution', 'fano_factor',
                 'voxel_mae', 'voxel_rmse', 'pmse_2_v2ce_adapted', 'pmse_4_v2ce_adapted']

# Voxel bin width in ms: voxel_* / pmse_* columns are read from the v2 report scored with exactly this width
# (run_voxel_metric_v2(_fast).py --voxel-bin-ms VOXEL_BIN_MS). Also used by the voxel figures (section 5).
VOXEL_BIN_MS = 10
VOXEL_CHANNEL = 'value'     # voxel_mae / voxel_rmse channel: 'value' (signed), 'value_on', 'value_off', 'value_aware', 'value_total'
# isi_distribution is saved as an ARRAY of intervals (no single score). Each statistic listed here becomes its OWN
# column in the tables, computed from the saved arrays (ms): 'median', 'mean', 'p90'.
#   ['median']          -> one column "ISI median (ms)"
#   ['median', 'mean']  -> two columns side by side
ISI_SUMMARY = ['median', 'mean']

# Rows of table 1 = method folder names in data_test_evt/<dataset>/<split>/ (order = row order)
# V2CE is left out: it only accepts 260x346 inputs and the evaluation now uses the original resolution.
BENCHMARKS = ['dvs_voltmeter', 'iebcs', 'fracevent', 'v2e', 'evis_core', 'evtween_v7_student']
OUR_MODEL = 'evtween_v7_student'   # highlighted model (bold rows, sorting of sequences in section 6)

# Rows of table 2: every <version>/model.pt (teacher) and <version>/student.pt (student) found here
CHECKPOINT_ROOT = Path('/u/marmary/airlab_v2e/evtween-tides-compare/benchmarks/event_generation/checkpoints/evtween_versions')
EXTRA_OUR_MODELS = []       # optional extra method names for table 2, e.g. ['evtween_v5.3_student']

# Paths (None = repository defaults)
DATA_ROOT = Path('/u/marmary/airlab_v2e/evtween/data')   # GT: DATA_ROOT/<dataset>/<split>/<sequence>/ev_{t,x,y,p}.npy
PRED_ROOT = None            # None -> <repo>/data_test_evt
REPORT_ROOT = None          # run_metric outputs; None -> [<repo>/reports, /u/marmary/airlab_v2e/evtween_bnchmark/reports] (missing ones ignored)
# Voxel/PMSE reports: <root>/<dataset>/<metric>/<run>/{config.json,results.csv}. Every run's config.json is checked:
# only runs with options.bin_ms == VOXEL_BIN_MS (whole-sequence window) are used, and only rows with status 'ok'.
VOXEL_REPORT_ROOTS = [Path('/u/marmary/airlab_v2e/evtween_bnchmark/reports_voxel_v2_ab')]
RUN_OVERRIDE = {}           # pin one run folder: {('bsergb', 'chamfer_full'): 'test-full-0123456789ab', ('eds', 'voxel_mae'): 'test-v2-...'}

# ----------------------------- Figures ---------------------------------------
VIS_SEQUENCE = None         # sequence for sections 3, 4, 5, 7 (None = first sequence that OUR_MODEL generated)
VIS_METHODS = BENCHMARKS    # models drawn in the figures (GT is always added first)

# Section 3: event image + x-t / y-t slices. Times are ms from the first frame of the sequence.
SPACETIME_T_START_MS, SPACETIME_T_STOP_MS = 0, 40
SLICE_Y = None              # x-t slice row (pixel). None = busiest GT row in the window
SLICE_X = None              # y-t slice column (pixel). None = busiest GT column in the window
SLICE_HALF_WIDTH = 0        # events with |y - SLICE_Y| <= this are drawn (0 = exactly one row)

# Section 4: event rate vs time
RATE_T_START_MS, RATE_T_STOP_MS = 0, None   # None stop = until the last frame
RATE_BIN_MS = 1.0

# Section 5: voxels
VOXEL_T_START_MS, VOXEL_T_STOP_MS = 0, 200  # window shown (None stop = whole sequence; long windows need RAM)
VOXEL_SHOW_BINS = 6         # number of time bins drawn in the voxel strip

# Section 6: event frames (defaults of the widgets)
FRAME_T_START_MS, FRAME_WINDOW_MS = 0, 20

# Section 7: 3D event cloud for 0..CLOUD_T_STOP_MS
CLOUD_T_START_MS, CLOUD_T_STOP_MS = 0, 30
CLOUD_METHODS = ['evtween_v7_student', 'v2e']   # one or more models drawn over GT
CLOUD_NEAR_PX = 2.0         # 'near GT' = nearest same-polarity GT event within this (x, y, t·1px/ms) distance
CLOUD_MAX_POINTS = 40000    # display cap per trace (deterministic subsample; distances use all events)

In [ ]:
import sys
import json
from datetime import datetime
from html import escape
from types import SimpleNamespace

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, HTML

ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / 'benchmarks/run_metric.py').is_file()), None)
if ROOT is None:
    raise RuntimeError('Open this notebook inside the evtween repository (benchmarks/run_metric.py not found).')
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from benchmarks import run_metric as runner
from benchmarks import voxel_metric_v2 as vox2

PRED = Path(PRED_ROOT) if PRED_ROOT else ROOT / 'data_test_evt'
_rr = REPORT_ROOT or [ROOT / 'reports', Path('/u/marmary/airlab_v2e/evtween_bnchmark/reports')]
REPORTS = list(dict.fromkeys(Path(p).resolve() for p in ([_rr] if isinstance(_rr, (str, Path)) else _rr)))
VOXEL_REPORTS = [Path(p) for p in (VOXEL_REPORT_ROOTS or [ROOT / 'reports_voxel_v2', ROOT / 'reports_voxel_v2_ab'])]
for _r in [*REPORTS, *VOXEL_REPORTS]:
    if not _r.is_dir(): print(f'(report root not found, ignored: {_r})')
GT_ROOT = Path(DATA_ROOT) if DATA_ROOT else runner.v2.old.setting('EVTWEEN_DATA', runner.v2.old.LAB / 'evtween/data')

# Output folder: reports/visualize_selected_metric_v2, or a date-time sub-folder when it already exists.
_base = ROOT / 'reports' / 'visualize_selected_metric_v2'
OUT_DIR = _base / datetime.now().strftime('%Y%m%d_%H%M%S') if _base.exists() else _base
OUT_DIR.mkdir(parents=True, exist_ok=True)
DATASET = VIS_DATASET        # dataset of the figures
if DATASETS == 'all':
    TABLE_DATASETS = sorted(q.parent.name for q in PRED.glob(f'*/{SPLIT}') if q.is_dir())
else:
    TABLE_DATASETS = [DATASETS] if isinstance(DATASETS, str) else list(DATASETS)
print(f'Repository : {ROOT}\nPredictions: {PRED}\nGT         : {GT_ROOT}\nVoxel runs : {", ".join(map(str, VOXEL_REPORTS))}'
      f'\nTables for : {", ".join(TABLE_DATASETS) or "(no dataset folder found)"}\nFigures for: {DATASET}\nSaving to  : {OUT_DIR}')

# ---------------------------------------------------------------- names
ALIASES = {'event_ratio': 'event_count_ratio', 'evebt_count': 'event_count', 'events_count': 'event_count',
           'event_counts': 'event_count', 'count_ratio': 'event_count_ratio', 'on_percent': 'on_percentage',
           'isi': 'isi_distribution', 'iei': 'iei_distance', 'iei_distribution': 'iei_distance',
           'fano': 'fano_factor', 'chamfer': 'chamfer_full', 'voxel_l1': 'voxel_relative_l1'}
VOXEL_METRICS = ('voxel_mae', 'voxel_rmse', 'voxel_relative_l1', 'pmse_2_v2ce_adapted', 'pmse_4_v2ce_adapted')
LABELS = {'chamfer_full': 'Chamfer full ↓', 'event_count': 'Event count', 'event_count_ratio': 'Event ratio ≈1',
          'on_percentage': 'ON (%)', 'iei_distance': 'IEI distance ↓',
          'fano_factor': 'Fano error ↓', 'voxel_mae': f'Voxel MAE ↓', 'voxel_rmse': f'Voxel RMSE ↓',
          'voxel_relative_l1': 'Voxel rel. L1 ↓', 'pmse_2_v2ce_adapted': 'PMSE-2 ↓', 'pmse_4_v2ce_adapted': 'PMSE-4 ↓'}
LOWER = {'chamfer_full', 'iei_distance', 'fano_factor', *VOXEL_METRICS}
ISI_STATS = [ISI_SUMMARY] if isinstance(ISI_SUMMARY, str) else list(ISI_SUMMARY)
ISI_FUNCS = {'median': np.median, 'mean': np.mean, 'p90': lambda a: np.percentile(a, 90)}
if set(ISI_STATS) - set(ISI_FUNCS):
    raise ValueError(f'ISI_SUMMARY must use {sorted(ISI_FUNCS)}')
ISI_COLUMNS = [f'isi_{s}' for s in ISI_STATS]                    # pseudo-metrics built from isi_distribution arrays
LABELS.update({f'isi_{s}': f'ISI {s} (ms)' for s in ISI_STATS})
DESCRIPTIVE = {'event_count', 'on_percentage', *ISI_COLUMNS}      # ranked by |value - GT|
METRICS = []
for _m in TABLE_METRICS:
    _m = ALIASES.get(_m.lower(), _m.lower())
    METRICS += ISI_COLUMNS if _m == 'isi_distribution' else [_m]
METRICS = list(dict.fromkeys(METRICS))
unknown = [m for m in METRICS if m not in LABELS]
if unknown:
    raise ValueError(f'Unknown TABLE_METRICS {unknown}; choose from {sorted(LABELS)} or aliases {sorted(ALIASES)}')


def pretty(method):
    """Readable row name: evtween_v7_student -> student v7."""
    if method == 'ground_truth':
        return 'GT'
    if method.startswith('evtween_'):
        _, version, mode = method.split('_', 2)
        return f'{mode} {version}'
    return {'dvs_voltmeter': 'DVS-Voltmeter', 'iebcs': 'IEBCS', 'fracevent': 'FracEvent', 'v2e': 'V2E',
            'evis_core': 'EVIS-core', 'v2ce': 'V2CE', 'v2v': 'V2V'}.get(method, method)


def safe(text):
    return ''.join(c if c.isalnum() or c in '-_.' else '_' for c in str(text))


def save_fig(fig, name):
    path = OUT_DIR / name
    if hasattr(fig, 'write_html'):
        fig.write_html(str(path), include_plotlyjs=True, config={'displaylogo': False, 'responsive': True})
    else:
        fig.savefig(path, dpi=150, bbox_inches='tight', facecolor='white')
    print(f'saved {path.relative_to(ROOT) if path.is_relative_to(ROOT) else path}')


def show_plotly(fig):
    display(HTML(fig.to_html(full_html=False, include_plotlyjs=True, config={'displaylogo': False, 'responsive': True})))


# ---------------------------------------------------------------- event streams
def method_folders():
    split_dir = PRED / DATASET / SPLIT
    return sorted(p for p in split_dir.glob('*') if p.is_dir()) if split_dir.is_dir() else []


def available_sequences(method=None):
    folders = [f for f in method_folders() if method is None or f.name == method]
    return sorted({p.name for f in folders for p in f.iterdir() if (p / 'events.npz').is_file()})


_SEQ_CACHE, _EV_CACHE = {}, {}


def sequence(name):
    """runner.Sequence for one sample (frame edges, crop metadata, aligned GT events in seconds)."""
    if name not in _SEQ_CACHE:
        args = SimpleNamespace(dataset=DATASET, split=SPLIT, data_root=DATA_ROOT)
        source = runner.source_for(name, GT_ROOT / DATASET / SPLIT, method_folders(), args)
        _SEQ_CACHE.clear(); _EV_CACHE.clear()          # keep one sequence in memory
        _SEQ_CACHE[name] = (runner.Sequence(source), args)
    return _SEQ_CACHE[name]


def load_streams(name, methods, verbose=True):
    """{'ground_truth': GT, method: events} with t in seconds from the first frame; missing models are reported."""
    seq, args = sequence(name)
    _, meta = seq.geometry
    streams = {'ground_truth': seq.gt}
    for method in dict.fromkeys(methods):
        key = (name, method)
        if key not in _EV_CACHE:
            file = PRED / DATASET / SPLIT / method / name / 'events.npz'
            try:
                if method == 'v2v':
                    raise ValueError('native V2V voxels have no individual events')
                if not file.is_file():
                    raise FileNotFoundError('no events.npz (events were not generated)')
                runner.verify_metadata(file.parent / 'input.json', meta, args, name)
                _EV_CACHE[key] = runner.read_events(file, meta)
            except Exception as exc:
                _EV_CACHE[key] = exc
        if isinstance(_EV_CACHE[key], Exception):
            if verbose:
                print(f'  {method}: skipped — {_EV_CACHE[key]}')
        else:
            streams[method] = _EV_CACHE[key]
    return streams, meta


def window(events, start_ms, stop_ms):
    """Events in [start, stop) ms (stream is time-sorted)."""
    a, b = np.searchsorted(events['t'], [start_ms / 1000, stop_ms / 1000])
    return {k: np.asarray(events[k][a:b]) for k in 'txyp'}


def default_sequence():
    if VIS_SEQUENCE:
        return VIS_SEQUENCE
    names = available_sequences(OUR_MODEL) or available_sequences()
    if not names:
        raise ValueError(f'No saved events.npz under {PRED / DATASET / SPLIT}')
    return names[0]


def event_rgb(ev, h, w, scale):
    """White background, ON red, OFF blue, both -> dark purple. scale = count that gives full color."""
    pix = ev['y'].astype(np.int64) * w + ev['x'].astype(np.int64)
    on = np.bincount(pix[ev['p'] > 0], minlength=h * w).reshape(h, w)
    off = np.bincount(pix[ev['p'] <= 0], minlength=h * w).reshape(h, w)
    a_on, a_off = np.clip(on / scale, 0, 1), np.clip(off / scale, 0, 1)
    rgb = np.ones((h, w, 3))
    rgb[..., 1] *= (1 - a_on) * (1 - a_off)
    rgb[..., 2] *= 1 - a_on
    rgb[..., 0] *= 1 - a_off
    return rgb


def count_scale(ev, h, w):
    pix = ev['y'].astype(np.int64) * w + ev['x'].astype(np.int64)
    counts = np.bincount(pix, minlength=h * w)
    counts = counts[counts > 0]
    return max(1.0, float(np.percentile(counts, 95))) if counts.size else 1.0


PALETTE = ['#2563eb', '#d97706', '#059669', '#dc2626', '#7c3aed', '#0891b2', '#be185d', '#4d7c0f', '#475569', '#9333ea']
def color_of(method, order):
    return '#111111' if method == 'ground_truth' else PALETTE[[m for m in order if m != 'ground_truth'].index(method) % len(PALETTE)]

In [ ]:
# Load saved scores (no scoring is launched). One long table: sequence, method, metric, value, status.
IGNORED_OPTIONS = {'device', 'workers', 'chunk_size', 'backend_version', 'torch_version', 'jobs'}


def _isi_path(folder, row):
    rel = row.get('array_path')
    path = folder / str(rel) if isinstance(rel, str) and rel else None
    return str(path) if path is not None and path.is_file() else ''


def _isi_stats(path):
    if not path:
        return {s: np.nan for s in ISI_STATS}
    with np.load(path) as z:
        isi = np.asarray(z['isi_ms']).ravel() if 'isi_ms' in z else np.array([])
    return {s: float(ISI_FUNCS[s](isi)) if isi.size else np.nan for s in ISI_STATS}


ISI_ARRAYS = {}   # dataset -> rows (sequence, method, path) of saved ISI arrays, used by the ISI CDF chart


def _read_runs(dataset, metric):
    """All compatible runs of one metric merged; per (sequence, method) prefer a finite score, then the newest run."""
    voxel = metric in VOXEL_METRICS
    roots = VOXEL_REPORTS if voxel else REPORTS
    runs = []
    for root in roots:
        for cfg_path in root.glob(f'{dataset}/{metric}/{SPLIT}-*/config.json'):
            csv_path = cfg_path.with_name('results.csv')
            if not csv_path.is_file():
                continue
            try:
                cfg = json.loads(cfg_path.read_text())
            except ValueError:
                continue
            if cfg.get('split', SPLIT) != SPLIT or cfg.get('dataset', dataset) != dataset:
                continue
            opts = cfg.get('options', {})
            if voxel:   # fixed-ms v2 protocol, requested bin width, whole-sequence window
                if (opts.get('protocol') != vox2.PROTOCOL or opts.get('bin_ms') is None
                        or not np.isclose(float(opts['bin_ms']), VOXEL_BIN_MS)
                        or float(opts.get('start_s') or 0) != 0 or opts.get('stop_s') is not None):
                    continue
                proto = json.dumps(opts, sort_keys=True)
            else:
                proto = json.dumps([{k: v for k, v in opts.items() if k not in IGNORED_OPTIONS},
                                    cfg.get('interval'), cfg.get('crop')], sort_keys=True, default=str)
            runs.append((csv_path.stat().st_mtime_ns, csv_path.parent, proto))
    pin = RUN_OVERRIDE.get((dataset, metric), RUN_OVERRIDE.get(metric))
    if pin:
        runs = [r for r in runs if r[1].name == pin]
        if not runs:
            raise ValueError(f'RUN_OVERRIDE for {dataset}/{metric} = {pin!r} not found')
    if not runs:
        return None, []
    runs.sort(key=lambda r: (r[0], str(r[1])))
    newest = runs[-1][2]
    used = [r for r in runs if r[2] == newest]
    skipped = [r[1].name for r in runs if r[2] != newest]
    if skipped:
        print(f'  {dataset}/{metric}: ignored runs with a different protocol {skipped} (pin one with RUN_OVERRIDE)')
    frames = []
    for order, (_, folder, _) in enumerate(used):
        frame = pd.read_csv(folder / 'results.csv')
        if metric in ('voxel_mae', 'voxel_rmse') and VOXEL_CHANNEL != 'value':
            frame['value'] = frame.get(VOXEL_CHANNEL, np.nan)
        if metric == 'isi_distribution':
            frame['path'] = [_isi_path(folder, r) if r['status'] == 'ok' else '' for r in frame.to_dict('records')]
            frame['value'] = np.where(frame['path'] != '', 1.0, np.nan)   # marks usable rows; stats added below
        frame['_order'] = order
        frames.append(frame)
    raw = pd.concat(frames, ignore_index=True)
    raw['value'] = pd.to_numeric(raw['value'], errors='coerce')
    ok = raw.status.isin(['ok', 'partial_support'] if metric == 'fano_factor' else ['ok'])
    raw['_usable'] = ok & np.isfinite(raw['value'])
    raw = (raw.sort_values(['_usable', '_order'], kind='stable')
              .drop_duplicates(['sequence', 'method'], keep='last'))
    raw.loc[~raw._usable, 'value'] = np.nan
    if metric == 'isi_distribution':
        ISI_ARRAYS[dataset] = raw.loc[raw._usable, ['sequence', 'method', 'path']]
        stats = pd.DataFrame([_isi_stats(p) for p in raw['path']], index=raw.index)
        return pd.concat([raw[['sequence', 'method', 'status']].assign(metric=f'isi_{s}', value=stats[s])
                          for s in ISI_STATS], ignore_index=True), [r[1] for r in used]
    return raw[['sequence', 'method', 'value', 'status']].assign(metric=metric), [r[1] for r in used]


def load_scores(dataset):
    parts, sources = [], {}
    for metric in dict.fromkeys('isi_distribution' if m in ISI_COLUMNS else m for m in METRICS):
        rows, folders = _read_runs(dataset, metric)
        sources[metric] = folders
        if rows is None:
            hint = (f'python benchmarks/run_voxel_metric_v2_fast.py {dataset} {SPLIT} --metric all --voxel-bin-ms {VOXEL_BIN_MS:g} --report-root {VOXEL_REPORTS[0]}'
                    if metric in VOXEL_METRICS else
                    f'python benchmarks/run_metric_fast.py {dataset} {SPLIT} --metric {metric}')
            print(f'  {metric}: NO saved report for {dataset}/{SPLIT}'
                  + (f' at {VOXEL_BIN_MS:g} ms bins' if metric in VOXEL_METRICS else '') + f'  ->  {hint}')
            continue
        parts.append(rows)
    scores = pd.concat(parts, ignore_index=True) if parts else pd.DataFrame(
        columns=['sequence', 'method', 'value', 'status', 'metric'])
    if SEQUENCES != 'all':
        wanted = [SEQUENCES] if isinstance(SEQUENCES, str) else list(SEQUENCES)
        missing = set(wanted) - set(scores.sequence)
        if missing:
            print(f'  sequences without any saved score: {sorted(missing)}')
        scores = scores[scores.sequence.isin(wanted)]
    return scores, sources


def summarize(scores, methods):
    """Per metric: mean per method over (matched) sequences, rank, n. GT is a reference row for descriptive metrics."""
    table, ranks, counts, notes = {}, {}, {}, []
    for metric in METRICS:
        rows = scores[scores.metric == metric]
        wide = rows.pivot_table(index='sequence', columns='method', values='value', aggfunc='first')
        cols = [m for m in methods if m in wide.columns and wide[m].notna().any()]
        for m in methods:
            if m not in cols:
                st = rows[rows.method == m].status.value_counts().to_dict()
                notes.append(f'{pretty(m):>16} | {metric:<20} | ' + ('not computed (no row in any report)' if not st
                             else 'no finite score: ' + ', '.join(f'{k}×{v}' for k, v in st.items())))
        need_gt = metric in DESCRIPTIVE
        has_gt = 'ground_truth' in wide.columns and wide['ground_truth'].notna().any()
        use = cols + (['ground_truth'] if need_gt and has_gt else [])
        wide = wide.reindex(columns=use)
        if MATCH_SEQUENCES and use:
            wide = wide.dropna()
        means = wide.mean()
        if metric == 'event_count_ratio':
            score = wide[cols].sub(1).abs().mean()
        elif metric in LOWER:
            score = means.reindex(cols)
        elif need_gt and has_gt:
            score = wide[cols].sub(wide['ground_truth'], axis=0).abs().mean()
        else:
            score = pd.Series(dtype=float)
        table[metric] = means
        ranks[metric] = score.rank(method='dense')
        counts[metric] = int(wide.notna().any(axis=1).sum()) if MATCH_SEQUENCES else int(wide.notna().sum().max() if use else 0)
    return table, ranks, counts, notes


RANK_COLORS = ['#164b80', '#4f86bd', '#9cc3e4', '#dcebf7']


def render_table(title, methods, scores, file_stem):
    table, ranks, counts, notes = summarize(scores, methods)
    rows = (['ground_truth'] if any('ground_truth' in table[m].index for m in METRICS) else []) + list(methods)
    out = pd.DataFrame({LABELS[m]: table[m].reindex(rows).to_numpy() for m in METRICS}, index=[pretty(r) for r in rows])
    html = [f"<h3>{escape(title)}</h3><div style='overflow-x:auto'><table style='border-collapse:collapse;font-size:13px'>"
            "<thead><tr><th style='text-align:left;padding:4px 8px'>Model</th>"]
    for m in METRICS:
        html.append(f"<th style='padding:4px 8px'>{escape(LABELS[m])}<br><small>n={counts[m]}</small></th>")
    html.append('</tr></thead><tbody>')
    for r in rows:
        if r == 'ground_truth' and not any(pd.notna(table[m].get(r)) for m in METRICS):
            continue
        bold = 'font-weight:700;' if r == OUR_MODEL else ''
        gt_style = 'font-style:italic;color:#555;' if r == 'ground_truth' else ''
        html.append(f"<tr><th style='text-align:left;padding:4px 8px;{bold}{gt_style}'>{escape(pretty(r))}</th>")
        for m in METRICS:
            v, rank = table[m].get(r, np.nan), ranks[m].get(r, np.nan)
            text = '—' if pd.isna(v) else (f'{v:,.0f}' if m == 'event_count' else f'{v:.4g}')
            style = f'padding:4px 8px;text-align:right;{bold}{gt_style}'
            if pd.notna(rank) and rank <= 4:
                style += f"background:{RANK_COLORS[int(rank) - 1]};color:{'white' if rank <= 2 else '#102b43'};"
            html.append(f"<td style='{style}'>{text}</td>")
        html.append('</tr>')
    html.append('</tbody></table></div>')
    display(HTML(''.join(html)))
    out.to_csv(OUT_DIR / f'{file_stem}.csv')
    (OUT_DIR / f'{file_stem}.html').write_text(''.join(html), encoding='utf-8')
    print(f'saved {file_stem}.csv / .html')
    if notes:
        print('Missing metrics (shown as —):\n  ' + '\n  '.join(notes))
    return out


seq_text = 'all test sequences' if SEQUENCES == 'all' else str(SEQUENCES)
SCORES_BY_DATASET = {}
for ds in TABLE_DATASETS:
    print(f'\n=== {ds}: loading saved reports ===')
    scores, sources = load_scores(ds)
    SCORES_BY_DATASET[ds] = scores
    for metric, folders in sources.items():
        if folders:
            print(f'  {metric:<20} <- ' + ', '.join(str(f) for f in folders))
    print(f'  {scores.sequence.nunique()} sequences with scores ({seq_text}); voxel columns at {VOXEL_BIN_MS:g} ms bins')
SCORES = SCORES_BY_DATASET.get(DATASET)
if SCORES is None:
    SCORES = load_scores(DATASET)[0]   # figures need the per-sequence scores of VIS_DATASET

## 1 · Benchmark table (one per dataset)
Rows = `BENCHMARKS` (folder names in `data_test_evt`), columns = `TABLE_METRICS`; voxel/PMSE columns come from the v2 report at `VOXEL_BIN_MS`. Each cell is the mean over sequences (`n` = matched sequences in that column). Blue = four best rank levels: lower is better for ↓ columns, |ratio − 1| for the event ratio, and |value − GT| for descriptive columns (GT row in italics). Missing scores are listed under the table.

In [ ]:
TABLE1 = {}
for ds, scores in SCORES_BY_DATASET.items():
    TABLE1[ds] = render_table(f'{ds}/{SPLIT} · benchmarks · mean over {seq_text}', BENCHMARKS, scores,
                              f'table1_benchmarks_{safe(ds)}_voxel{VOXEL_BIN_MS:g}ms')

## 2 · Our models (all teacher / student checkpoints)
Rows are discovered from `CHECKPOINT_ROOT/<version>/model.pt` (teacher → `evtween_<version>_teacher`) and `student.pt` (student → `evtween_<version>_student`), plus `EXTRA_OUR_MODELS`. Columns are identical to table 1. A model whose events or metrics were never computed is reported below the table.

In [ ]:
import re

def version_key(name):
    return [int(p) if p.isdigit() else p for p in re.split(r'(\d+)', name)]

def discover_our_models():
    found = []
    if CHECKPOINT_ROOT.is_dir():
        for folder in sorted((p for p in CHECKPOINT_ROOT.iterdir() if p.is_dir()), key=lambda p: version_key(p.name)):
            for file, mode in (('model.pt', 'teacher'), ('student.pt', 'student')):
                if (folder / file).is_file():
                    found.append(f'evtween_{folder.name}_{mode}')
            if not any((folder / f).is_file() for f in ('model.pt', 'student.pt')):
                print(f'  {folder.name}: no model.pt / student.pt')
    else:
        print(f'Checkpoint folder not reachable: {CHECKPOINT_ROOT}\n  -> using evtween_* methods found in reports and data_test_evt')
        found = sorted({str(m) for sc in SCORES_BY_DATASET.values() for m in sc.method.unique() if str(m).startswith('evtween_')}
                       | {q.parent.name for q in PRED.glob(f'*/{SPLIT}/evtween_*/')}, key=version_key)
    found += [m for m in EXTRA_OUR_MODELS if m not in found]
    for ds in TABLE_DATASETS:
        generated = {p.name for p in (PRED / ds / SPLIT).glob('*') if p.is_dir()}
        for m in found:
            if m not in generated:
                print(f'  {ds}: {m} has no generated events in data_test_evt (generate + score it to fill the row)')
    return found

OUR_MODELS = discover_our_models()
print('Our models:', ', '.join(pretty(m) for m in OUR_MODELS))
TABLE2 = {}
for ds, scores in SCORES_BY_DATASET.items():
    TABLE2[ds] = render_table(f'{ds}/{SPLIT} · our models · mean over {seq_text}', OUR_MODELS, scores,
                              f'table2_our_models_{safe(ds)}_voxel{VOXEL_BIN_MS:g}ms')

## 2b · ISI distribution (cumulative)
Same chart as the stride/window experiments: each curve is the equally weighted mean of the per-sequence empirical CDFs of the saved `isi_distribution` arrays (`isi_ms`) — `(k/N)` = sequences contributing. GT is the black dashed line. The x axis is log-like with a finite origin so zero intervals (simultaneous events) stay visible. One chart per dataset for the benchmarks and one for our models; hover for the fraction at each ISI, click the legend to hide a model.

In [ ]:
import plotly.graph_objects as go

ISI_GRID_POINTS = 200
_CDF_CACHE = {}

def isi_axis(ms):
    return np.log10(1 + np.asarray(ms, dtype=float) / 0.001)


def load_isi(path):
    if path not in _CDF_CACHE:
        with np.load(path, allow_pickle=False) as z:
            _CDF_CACHE[path] = np.sort(np.asarray(z['isi_ms'], dtype=np.float64).ravel())
    return _CDF_CACHE[path]


def isi_cdf_figure(dataset, methods, title):
    rows = ISI_ARRAYS.get(dataset)
    if rows is None:
        _read_runs(dataset, 'isi_distribution')          # ISI not in TABLE_METRICS: load its arrays anyway
        rows = ISI_ARRAYS.get(dataset)
    if rows is None or rows.empty:
        print(f'{dataset}: no saved isi_distribution arrays  ->  python benchmarks/run_metric_fast.py {dataset} {SPLIT} --metric isi_distribution')
        return None
    if SEQUENCES != 'all':
        rows = rows[rows.sequence.isin([SEQUENCES] if isinstance(SEQUENCES, str) else list(SEQUENCES))]
    order = ['ground_truth', *methods]
    rows = rows[rows.method.isin(order)]
    n_total = rows.sequence.nunique()
    arrays = {(r.sequence, r.method): load_isi(r.path) for r in rows.itertuples()}
    upper = max([a[-1] for a in arrays.values() if a.size] + [1e-6])
    grid = np.r_[0, np.geomspace(1e-6, upper, ISI_GRID_POINTS)]
    fig = go.Figure()
    for m in order:
        curves = [np.searchsorted(a, grid, side='right') / a.size
                  for (s, mm), a in arrays.items() if mm == m and a.size]
        if not curves:
            if m != 'ground_truth':
                print(f'  {dataset}: {pretty(m)} has no saved ISI array')
            continue
        gt = m == 'ground_truth'
        fig.add_scatter(x=isi_axis(grid), y=np.mean(curves, axis=0), customdata=grid, mode='lines',
                        name=f'{pretty(m)} ({len(curves)}/{n_total})',
                        line=dict(color='#161616', dash='dash', width=2.5) if gt else
                             dict(color=color_of(m, order), width=3 if m == OUR_MODEL else 1.5),
                        hovertemplate='%{customdata:.5g} ms · fraction %{y:.4f}<extra>%{fullData.name}</extra>')
    ticks = [v for v in (0, .001, .01, .1, 1, 10, 100, 1000, 10000, 100000) if v <= upper * 10]
    fig.update_layout(template='plotly_white', height=480, title=title,
                      xaxis=dict(title='ISI (ms)', tickvals=isi_axis(ticks), ticktext=[f'{v:g}' for v in ticks]),
                      yaxis=dict(title='Mean cumulative fraction', range=[0, 1.02]))
    return fig


for ds in TABLE_DATASETS:
    for group, methods in (('benchmarks', BENCHMARKS), ('our_models', OUR_MODELS)):
        fig = isi_cdf_figure(ds, methods, f'ISI cumulative distribution · {ds}/{SPLIT} · {group.replace("_", " ")} · {seq_text}')
        if fig is not None and fig.data:
            show_plotly(fig)
            save_fig(fig, f'isi_cdf_{safe(ds)}_{group}.html')

## 3 · Event image + x-t / y-t slices
Window `SPACETIME_T_START_MS`–`SPACETIME_T_STOP_MS` (ms from the first frame), slices at `SLICE_Y` / `SLICE_X` (green lines; `None` = busiest GT row/column). One row per model, GT first, always three panels per row: event image, x-t, y-t. Saved: one figure with every model plus one file per model (GT row + that model), named `spacetime_<dataset>_<sequence>_<model>_<start>-<stop>ms.png`. Titles show the saved per-sequence Chamfer score.

The interactive chart under it puts **all models on one x-t / y-t chart** over GT (black): toggle models in the legend, switch ON / OFF with the buttons.

In [ ]:
# Section 3 — event image + x-t + y-t slices (one row per model, GT first, always 3 panels per row)
def per_sequence_score(name, method, metric='chamfer_full'):
    hit = SCORES[(SCORES.sequence == name) & (SCORES.method == method) & (SCORES.metric == metric)]
    return float(hit.value.iloc[0]) if len(hit) and pd.notna(hit.value.iloc[0]) else None


def spacetime_rows(name, methods, t0, t1, slice_y=None, slice_x=None, half=0):
    streams, meta = load_streams(name, methods)
    h, w = meta['height'], meta['width']
    cut = {m: window(ev, t0, t1) for m, ev in streams.items()}
    gt = cut['ground_truth']
    if slice_y is None:
        slice_y = int(np.argmax(np.bincount(gt['y'], minlength=h))) if gt['t'].size else h // 2
    if slice_x is None:
        slice_x = int(np.argmax(np.bincount(gt['x'], minlength=w))) if gt['t'].size else w // 2
    return cut, meta, slice_y, slice_x


def draw_row(axes, ev, label, meta, scale, t0, t1, sy, sx, half):
    h, w = meta['height'], meta['width']
    a0, a1, a2 = axes
    a0.imshow(event_rgb(ev, h, w, scale), interpolation='nearest')
    a0.axhline(sy, color='green', lw=0.8); a0.axvline(sx, color='green', lw=0.8)
    a0.set_title(label, fontsize=10); a0.axis('off')
    t_ms = ev['t'] * 1000
    for ax, keep, coord, axis_name, size in ((a1, np.abs(ev['y'] - sy) <= half, ev['x'], 'x', w),
                                            (a2, np.abs(ev['x'] - sx) <= half, ev['y'], 'y', h)):
        on = keep & (ev['p'] > 0); off = keep & (ev['p'] <= 0)
        ax.scatter(coord[off], t_ms[off], s=1.2, c='blue', linewidths=0)
        ax.scatter(coord[on], t_ms[on], s=1.2, c='red', linewidths=0)
        ax.set_xlim(0, size); ax.set_ylim(t0, t1)
        ax.set_xlabel(f'{axis_name} (px)'); ax.set_ylabel('t (ms)')
        ax.set_title(f'{axis_name}-t at {"y" if axis_name == "x" else "x"}={sy if axis_name == "x" else sx}'
                     + (f' ±{half}' if half else ''), fontsize=10)


def spacetime_figure(name, rows, cut, meta, t0, t1, sy, sx, half):
    scale = count_scale(cut['ground_truth'], meta['height'], meta['width'])
    fig, axes = plt.subplots(len(rows), 3, figsize=(13, 3.6 * len(rows)), squeeze=False)
    for ax_row, m in zip(axes, rows):
        ev = cut[m]
        score = per_sequence_score(name, m) if m != 'ground_truth' else None
        label = f'{pretty(m)}: {ev["t"].size:,} events' + (f' · chamfer {score:.3g}' if score is not None else '')
        draw_row(ax_row, ev, label, meta, scale, t0, t1, sy, sx, half)
    fig.suptitle(f'{DATASET}/{name} · {t0:g}–{t1:g} ms', y=1.0, fontsize=12)
    fig.tight_layout()
    return fig


ST_SEQ = default_sequence()
t0, t1 = SPACETIME_T_START_MS, SPACETIME_T_STOP_MS
ST_CUT, ST_META, ST_Y, ST_X = spacetime_rows(ST_SEQ, VIS_METHODS, t0, t1, SLICE_Y, SLICE_X, SLICE_HALF_WIDTH)
ST_ROWS = ['ground_truth'] + [m for m in VIS_METHODS if m in ST_CUT]
print(f'{ST_SEQ}: slices y={ST_Y}, x={ST_X}, window {t0:g}–{t1:g} ms')

fig = spacetime_figure(ST_SEQ, ST_ROWS, ST_CUT, ST_META, t0, t1, ST_Y, ST_X, SLICE_HALF_WIDTH)
save_fig(fig, f'spacetime_{safe(DATASET)}_{safe(ST_SEQ)}_all_models_{t0:g}-{t1:g}ms.png')
plt.show()
for m in ST_ROWS[1:]:   # one file per model: GT row + model row
    f = spacetime_figure(ST_SEQ, ['ground_truth', m], ST_CUT, ST_META, t0, t1, ST_Y, ST_X, SLICE_HALF_WIDTH)
    save_fig(f, f'spacetime_{safe(DATASET)}_{safe(ST_SEQ)}_{safe(m)}_{t0:g}-{t1:g}ms.png')
    plt.close(f)

In [ ]:
# Interactive: all models on ONE x-t / y-t chart over GT (same window and slices as above).
# Legend click = hide/show a model · buttons = polarity · drag = zoom · double-click = reset.
import plotly.graph_objects as go  # (also imported in 2b)
from plotly.subplots import make_subplots

def spacetime_overlay(name, rows, cut, meta, t0, t1, sy, sx, half):
    fig = make_subplots(rows=1, cols=2, horizontal_spacing=0.07,
                        subplot_titles=(f'x-t at y={sy}' + (f' ±{half}' if half else ''),
                                        f'y-t at x={sx}' + (f' ±{half}' if half else '')))
    polarity_of = []
    for m in rows:
        ev, color = cut[m], color_of(m, rows)
        gt = m == 'ground_truth'
        for pol, keep_p, symbol in (('ON', ev['p'] > 0, 'circle'), ('OFF', ev['p'] <= 0, 'x')):
            for col, (keep_s, coord) in enumerate(((np.abs(ev['y'] - sy) <= half, ev['x']),
                                                   (np.abs(ev['x'] - sx) <= half, ev['y'])), start=1):
                k = keep_s & keep_p
                fig.add_trace(go.Scattergl(
                    x=coord[k], y=ev['t'][k] * 1000, mode='markers', name=pretty(m), legendgroup=m,
                    showlegend=(pol == 'ON' and col == 1),
                    marker=dict(size=7 if gt else 5, color=color, symbol=symbol, opacity=0.35 if gt else 0.85,
                                line=dict(width=1, color=color)),
                    hovertemplate=f'{pretty(m)} {pol}<br>%{{x}} px · %{{y:.3f}} ms<extra></extra>'), row=1, col=col)
                polarity_of.append(pol)
    vis = lambda allowed: [p in allowed for p in polarity_of]
    fig.update_layout(template='plotly_white', height=640,
                      title=f'{DATASET}/{name} · {t0:g}–{t1:g} ms · GT (black) vs models — ● ON, ✕ OFF',
                      legend=dict(orientation='h', y=-0.12), margin=dict(t=110),
                      updatemenus=[dict(type='buttons', direction='right', x=1, y=1.17, xanchor='right', buttons=[
                          dict(label='ON + OFF', method='restyle', args=['visible', vis({'ON', 'OFF'})]),
                          dict(label='ON only', method='restyle', args=['visible', vis({'ON'})]),
                          dict(label='OFF only', method='restyle', args=['visible', vis({'OFF'})])])])
    fig.update_xaxes(title_text='x (px)', range=[0, meta['width']], row=1, col=1)
    fig.update_xaxes(title_text='y (px)', range=[0, meta['height']], row=1, col=2)
    fig.update_yaxes(title_text='t (ms)', range=[t0, t1])
    return fig

fig = spacetime_overlay(ST_SEQ, ST_ROWS, ST_CUT, ST_META, t0, t1, ST_Y, ST_X, SLICE_HALF_WIDTH)
show_plotly(fig)
save_fig(fig, f'spacetime_overlay_{safe(DATASET)}_{safe(ST_SEQ)}_{t0:g}-{t1:g}ms.html')

## 4 · Events vs time (interactive)
Same chart as `watch_voxel_metrics_v2` (`benchmarks/event_rate_views_v2.make_rate_figures`): ON / OFF panels and total ON+OFF, GT in black. Window: `RATE_T_START_MS`–`RATE_T_STOP_MS` (None = up to the last frame), bin width `RATE_BIN_MS`. Zoom, Reset zoom, Linear / Log Y, legend toggles.

In [ ]:
# Section 4 — Events vs time: the same interactive chart as watch_voxel_metrics_v2 (benchmarks/event_rate_views_v2.py)
from benchmarks import event_rate_views_v2 as rates

R_SEQ = default_sequence()
r_streams, r_meta = load_streams(R_SEQ, VIS_METHODS, verbose=False)
frames, _ = sequence(R_SEQ)[0].geometry
r_start = RATE_T_START_MS / 1000
r_stop = float(frames[-1]) if RATE_T_STOP_MS is None else min(RATE_T_STOP_MS / 1000, float(frames[-1]))
if r_stop <= r_start:
    raise ValueError('Choose RATE_T_START_MS < RATE_T_STOP_MS inside the sequence')
named = {('Ground truth' if m == 'ground_truth' else pretty(m)): ev for m, ev in r_streams.items()}
title = f'{DATASET}/{SPLIT}/{R_SEQ} · {r_start*1000:g}–{r_stop*1000:g} ms · {RATE_BIN_MS:g} ms bins'
print(title)
onoff, total, _ = rates.make_rate_figures(named, r_start, r_stop, RATE_BIN_MS, title)
for label, fig in (('on-off', onoff), ('total', total)):
    show_plotly(fig)
    save_fig(fig, f'event_rate_{label}_{safe(DATASET)}_{safe(R_SEQ)}_{r_start*1000:g}-{r_stop*1000:g}ms_bin{RATE_BIN_MS:g}ms.html')

## 5 · Voxels
Hard-count voxels with the v2 fixed-ms bins (`VOXEL_BIN_MS`) over `VOXEL_T_START_MS`–`VOXEL_T_STOP_MS`. Display-only; scores in the tables are untouched.
- **Voxel strip**: signed voxels (ON − OFF, red/blue on white) of GT and every model at the same bins, models sorted best → worst by voxel MAE in this window.
- **Error atlas**: per-pixel Σ|P − G| over the window on one shared scale — white means the model matches GT, dark means it misses or invents events. Three panels per row, GT activity first.
- **Error over time** (interactive): voxel MAE per bin, with GT activity on the right axis.

In [ ]:
# Section 5 — Voxels at VOXEL_BIN_MS (same fixed-ms bins as the v2 voxel metrics; display only, scores are not changed)
#   (a) voxel strip   : signed voxels (ON − OFF) of GT and every model at the same time bins
#   (b) error atlas   : per pixel Σ_bins |P − G| over the window, models sorted best → worst (white = no error)
#   (c) interactive   : voxel MAE of every model in each time bin
#
# Order of the models (best → worst) in all three figures:
#   'pmse_2_v2ce_adapted' or 'pmse_4_v2ce_adapted' -> saved per-sequence score of V_SEQ (v2 report at VOXEL_BIN_MS,
#                                                    scored over the WHOLE sequence; needs the metric in TABLE_METRICS)
#   'mae'                                          -> voxel MAE computed here, in the displayed window only
RANK_BY = 'pmse_2_v2ce_adapted'

V_SEQ = default_sequence()
v_streams, v_meta = load_streams(V_SEQ, VIS_METHODS, verbose=False)
H, W = v_meta['height'], v_meta['width']
frames, _ = sequence(V_SEQ)[0].geometry
v_start = VOXEL_T_START_MS / 1000
v_stop = float(frames[-1]) if VOXEL_T_STOP_MS is None else min(VOXEL_T_STOP_MS / 1000, float(frames[-1]))
V_EDGES = vox2.time_edges(v_start, v_stop, VOXEL_BIN_MS)
B = len(V_EDGES) - 1
print(f'{V_SEQ}: {B} bins of {VOXEL_BIN_MS:g} ms over {v_start*1000:g}–{v_stop*1000:g} ms (last bin {np.diff(V_EDGES)[-1]*1000:.3g} ms), {H}×{W} px')


def signed_voxels(ev):
    t = np.asarray(ev['t'])
    keep = (t >= V_EDGES[0]) & (t < V_EDGES[-1])
    b = np.searchsorted(V_EDGES, t[keep], side='right') - 1
    ids = (b * H + np.asarray(ev['y'])[keep]) * W + np.asarray(ev['x'])[keep]
    sign = np.where(np.asarray(ev['p'])[keep] > 0, 1.0, -1.0)
    return np.bincount(ids, weights=sign, minlength=B * H * W).reshape(B, H, W).astype(np.float32)


GRIDS = {m: signed_voxels(ev) for m, ev in v_streams.items()}
G = GRIDS['ground_truth']
MODELS = [m for m in VIS_METHODS if m in GRIDS]
ERR = {m: np.abs(GRIDS[m] - G) for m in MODELS}
MAE = {m: float(ERR[m].mean()) for m in MODELS}

# ---- ranking score (lower = better)
if RANK_BY == 'mae':
    RANK_NAME = 'MAE (window)'
    RANK_SCORE = dict(MAE)
else:
    if RANK_BY not in METRICS:
        print(f'{RANK_BY} is not in TABLE_METRICS, so it was not loaded -> add it there and re-run the report cell')
    RANK_NAME = {'pmse_2_v2ce_adapted': 'PMSE-2', 'pmse_4_v2ce_adapted': 'PMSE-4'}.get(RANK_BY, RANK_BY)
    saved = {m: per_sequence_score(V_SEQ, m, RANK_BY) for m in MODELS}
    missing = [pretty(m) for m, v in saved.items() if v is None]
    if missing:
        print(f'No saved {RANK_NAME} for {V_SEQ} at {VOXEL_BIN_MS:g} ms bins: {", ".join(missing)} -> placed last')
    RANK_SCORE = {m: (np.inf if v is None else v) for m, v in saved.items()}
RANKED = sorted(MODELS, key=lambda m: (RANK_SCORE[m], MAE[m]))   # ties / missing -> window MAE


def score_text(m, digits=3):
    """'PMSE-2 0.123 · MAE 0.0456' (or only MAE when ranking by MAE)."""
    mae = f'MAE {MAE[m]:.{digits + 1}f}'
    if RANK_BY == 'mae':
        return mae
    v = RANK_SCORE[m]
    return f'{RANK_NAME} {"—" if not np.isfinite(v) else f"{v:.{digits}g}"} · {mae}'


print(f'order by {RANK_NAME}: ' + ', '.join(f'{pretty(m)} ({score_text(m)})' for m in RANKED))

# (a) voxel strip — rows: GT + models (best first); columns: evenly spaced, GT-busy bins
from matplotlib.colors import TwoSlopeNorm
busy = np.abs(G).reshape(B, -1).sum(1)
candidates = np.flatnonzero(busy > 0) if np.any(busy > 0) else np.arange(B)
cols = candidates[np.linspace(0, len(candidates) - 1, min(VOXEL_SHOW_BINS, len(candidates))).round().astype(int)]
vmax = max(1.0, float(np.percentile(np.abs(G[cols][G[cols] != 0]), 98)) if np.any(G[cols]) else 1.0)
norm = TwoSlopeNorm(vmin=-vmax, vcenter=0, vmax=vmax)
rows = ['ground_truth'] + RANKED
fig, axes = plt.subplots(len(rows), len(cols), figsize=(2.3 * len(cols) + 1.6, 1.9 * len(rows)), squeeze=False)
for i, m in enumerate(rows):
    for j, b in enumerate(cols):
        ax = axes[i, j]
        im = ax.imshow(GRIDS[m][b], cmap='bwr', norm=norm, interpolation='nearest')
        ax.set_xticks([]); ax.set_yticks([])
        if i == 0:
            ax.set_title(f'{V_EDGES[b]*1000:.0f}–{V_EDGES[b+1]*1000:.0f} ms', fontsize=9)
        if j == 0:
            lab = pretty(m) if m == 'ground_truth' else f'#{RANKED.index(m) + 1} {pretty(m)}\n' + score_text(m).replace(' · ', '\n')
            ax.set_ylabel(lab, fontsize=8.5, fontweight='bold' if m == OUR_MODEL else 'normal')
fig.colorbar(im, ax=axes, shrink=0.5, label='ON − OFF events / voxel')
fig.suptitle(f'Signed voxels · {DATASET}/{V_SEQ} · {VOXEL_BIN_MS:g} ms bins (red = ON, blue = OFF) · sorted by {RANK_NAME}', y=1.0)
save_fig(fig, f'voxel_strip_{safe(DATASET)}_{safe(V_SEQ)}_{v_start*1000:g}-{v_stop*1000:g}ms_bin{VOXEL_BIN_MS:g}ms_by_{safe(RANK_BY)}.png')
plt.show()

# (b) error atlas — 3 panels per row; first panel = GT activity for reference
emaps = {m: ERR[m].sum(0) for m in RANKED}
emax = max(1.0, float(np.percentile(np.concatenate([e[e > 0] for e in emaps.values()] or [np.ones(1)]), 98)))
panels = ['ground_truth'] + RANKED
nrow = int(np.ceil(len(panels) / 3))
fig, axes = plt.subplots(nrow, 3, figsize=(13, 3.9 * nrow), squeeze=False)
for ax in axes.flat:
    ax.axis('off')
for ax, m in zip(axes.flat, panels):
    if m == 'ground_truth':
        act = np.abs(G).sum(0)
        ax.imshow(act, cmap='Greys', vmin=0, vmax=max(1, np.percentile(act[act > 0], 98) if act.any() else 1))
        ax.set_title('GT activity  Σ|G|', fontsize=10)
    else:
        imr = ax.imshow(emaps[m], cmap='magma_r', vmin=0, vmax=emax)
        rank = RANKED.index(m) + 1
        ax.set_title(f'#{rank} {pretty(m)} · {score_text(m)}', fontsize=10,
                     fontweight='bold' if m == OUR_MODEL else 'normal',
                     color='#164b80' if m == OUR_MODEL else 'black')
fig.colorbar(imr, ax=axes, shrink=0.6, label='Σ over bins |P − G|  (events)')
fig.suptitle(f'Voxel error atlas · {DATASET}/{V_SEQ} · {v_start*1000:g}–{v_stop*1000:g} ms · {VOXEL_BIN_MS:g} ms bins '
             f'(white = matches GT) · sorted by {RANK_NAME}', y=1.0)
save_fig(fig, f'voxel_error_atlas_{safe(DATASET)}_{safe(V_SEQ)}_{v_start*1000:g}-{v_stop*1000:g}ms_bin{VOXEL_BIN_MS:g}ms_by_{safe(RANK_BY)}.png')
plt.show()

# (c) interactive voxel MAE per time bin (legend in the same order)
centers = (V_EDGES[:-1] + V_EDGES[1:]) / 2 * 1000
fig = go.Figure()
for m in RANKED:
    fig.add_trace(go.Scatter(x=centers, y=ERR[m].reshape(B, -1).mean(1), mode='lines', name=f'{pretty(m)} ({score_text(m)})',
                             line=dict(color=color_of(m, MODELS), width=3 if m == OUR_MODEL else 1.5, shape='hvh')))
fig.add_trace(go.Scatter(x=centers, y=np.abs(G).reshape(B, -1).mean(1), mode='lines', name='GT |signed| per voxel',
                         line=dict(color='#999', dash='dot'), yaxis='y2'))
fig.update_layout(template='plotly_white', height=480, hovermode='x unified',
                  title=f'Voxel MAE per {VOXEL_BIN_MS:g} ms bin · {DATASET}/{V_SEQ} (lower = closer to GT) · legend sorted by {RANK_NAME}',
                  xaxis_title='time (ms)', yaxis_title='|P − G| per voxel',
                  yaxis2=dict(overlaying='y', side='right', title='GT activity', showgrid=False),
                  legend=dict(orientation='h', y=-0.2))
show_plotly(fig)
save_fig(fig, f'voxel_mae_over_time_{safe(DATASET)}_{safe(V_SEQ)}_{v_start*1000:g}-{v_stop*1000:g}ms_bin{VOXEL_BIN_MS:g}ms_by_{safe(RANK_BY)}.html')
del ERR

## 6 · Event frames on white background — choose the sequence
Dropdown labels show where `OUR_MODEL` ranks on Chamfer for each sequence (best samples first). Set start / window length (ms) and the models, then **Draw**. The saved per-sequence scores of every model are listed under the frames.

In [ ]:
# Section 6 — Event frames on white background (ON red, OFF blue), GT first, 3 panels per row.
# Pick a sequence in the dropdown: it is labelled with OUR_MODEL's rank among the selected models on chamfer_full,
# so the samples where our model wins are easy to find. The per-sequence scores of all models are shown below the frames.
import ipywidgets as widgets
from IPython.display import clear_output

def sequence_ranking(metric='chamfer_full'):
    rows = SCORES[(SCORES.metric == metric) & SCORES.method.isin(VIS_METHODS)]
    wide = rows.pivot_table(index='sequence', columns='method', values='value', aggfunc='first')
    if OUR_MODEL not in wide.columns:
        return pd.DataFrame(columns=['rank', 'of', 'margin'])
    rank = wide.rank(axis=1, method='min')[OUR_MODEL]
    best_other = wide.drop(columns=OUR_MODEL).min(axis=1)
    out = pd.DataFrame({'rank': rank, 'of': wide.notna().sum(axis=1), 'ours': wide[OUR_MODEL],
                        'best_other': best_other, 'margin': best_other - wide[OUR_MODEL]})
    return out.dropna(subset=['rank']).sort_values(['rank', 'margin'], ascending=[True, False])


def per_sequence_table(name):
    rows = SCORES[(SCORES.sequence == name) & SCORES.method.isin(['ground_truth', *VIS_METHODS])]
    wide = rows.pivot_table(index='method', columns='metric', values='value', aggfunc='first')
    wide = wide.reindex(index=[m for m in ['ground_truth', *VIS_METHODS] if m in wide.index],
                        columns=[m for m in METRICS if m in wide.columns])
    wide.index = [pretty(m) for m in wide.index]
    wide.columns = [LABELS[c] for c in wide.columns]
    return wide


def draw_frames(name, start_ms, length_ms, methods, save=False):
    streams, meta = load_streams(name, methods)
    h, w = meta['height'], meta['width']
    stop_ms = start_ms + length_ms
    cut = {m: window(ev, start_ms, stop_ms) for m, ev in streams.items()}
    scale = count_scale(cut['ground_truth'], h, w)
    panels = ['ground_truth'] + [m for m in methods if m in cut]
    nrow = int(np.ceil(len(panels) / 3))
    fig, axes = plt.subplots(nrow, 3, figsize=(13, 13 * h / w / 3 * nrow + 0.6 * nrow), squeeze=False)
    for ax in axes.flat:
        ax.axis('off')
    for ax, m in zip(axes.flat, panels):
        ax.imshow(event_rgb(cut[m], h, w, scale), interpolation='nearest')
        score = per_sequence_score(name, m) if m != 'ground_truth' else None
        ax.set_title(f'{pretty(m)} · {cut[m]["t"].size:,} ev' + (f' · chamfer {score:.3g}' if score is not None else ''),
                     fontsize=10, fontweight='bold' if m == OUR_MODEL else 'normal')
    fig.suptitle(f'{DATASET}/{name} · {start_ms:g}–{stop_ms:g} ms', y=1.0)
    fig.tight_layout()
    if save:
        save_fig(fig, f'event_frames_{safe(DATASET)}_{safe(name)}_{start_ms:g}-{stop_ms:g}ms.png')
    plt.show()
    table = per_sequence_table(name)
    if len(table):
        display(table.style.format('{:.4g}', na_rep='—').set_caption(f'Saved scores for {name}'))


ranking = sequence_ranking()
all_seqs = available_sequences()
label = {s: (f'{s}  ·  {pretty(OUR_MODEL)} rank {int(ranking.loc[s, "rank"])}/{int(ranking.loc[s, "of"])}'
             if s in ranking.index else s) for s in all_seqs}
ordered = list(ranking.index.intersection(all_seqs)) + [s for s in all_seqs if s not in ranking.index]
seq_dd = widgets.Dropdown(options=[(label[s], s) for s in ordered], value=default_sequence() if default_sequence() in ordered else (ordered[0] if ordered else None),
                          description='Sequence:', layout=widgets.Layout(width='520px'))
start_w = widgets.FloatText(value=FRAME_T_START_MS, description='Start (ms):')
length_w = widgets.FloatText(value=FRAME_WINDOW_MS, description='Window (ms):')
methods_w = widgets.SelectMultiple(options=[(pretty(m), m) for m in dict.fromkeys([*VIS_METHODS, *OUR_MODELS])],
                                   value=tuple(VIS_METHODS), description='Models:', rows=8)
save_w = widgets.Checkbox(value=True, description='Save PNG')
go_w = widgets.Button(description='Draw', icon='image', button_style='primary')
out_w = widgets.Output()

def _draw(_=None):
    with out_w:
        clear_output(wait=True)
        try:
            draw_frames(seq_dd.value, start_w.value, length_w.value, list(methods_w.value), save_w.value)
        except Exception as exc:
            print(f'Cannot draw: {exc}')
go_w.on_click(_draw)
display(widgets.VBox([seq_dd, widgets.HBox([start_w, length_w]), widgets.HBox([methods_w, widgets.VBox([save_w, go_w])]), out_w]))
if len(ranking):
    print(f'Sequences where {pretty(OUR_MODEL)} is best on chamfer_full (margin = best other − ours):')
    display(ranking[ranking['rank'] == 1].head(15).style.format({'ours': '{:.4g}', 'best_other': '{:.4g}', 'margin': '{:.4g}', 'rank': '{:.0f}', 'of': '{:.0f}'}))
_draw()

## 7 · 3D event cloud: GT + selected models
Events in `CLOUD_T_START_MS`–`CLOUD_T_STOP_MS` as a point cloud (x, t, y). GT is the grey base cloud; every model in `CLOUD_METHODS` is overlaid in its own color. Three figures: **ON**, **OFF**, **ON+OFF**. In each figure, the buttons switch between **All events**, **Near GT only** (nearest same-polarity GT event ≤ `CLOUD_NEAR_PX` in x, y, t·1 px/ms — the Chamfer space) and **Far from GT**. The table gives the share of model events near GT and the share of GT covered by the model.

In [ ]:
# Section 7 — 3D event cloud (x, t, y) for CLOUD_T_START_MS..CLOUD_T_STOP_MS.
# GT = grey base cloud (always present). Each model in CLOUD_METHODS is drawn on top in its own color.
# Three figures: ON events, OFF events, ON+OFF (● ON, ◆ OFF). Buttons inside each figure switch the model events between
#   All · Near GT (nearest same-polarity GT event ≤ CLOUD_NEAR_PX) · Far from GT (> CLOUD_NEAR_PX).
# Distance = Euclidean in (x px, y px, t ms × 1 px/ms), the same space as chamfer_full. Legend click hides a model.
from scipy.spatial import cKDTree

C_SEQ = default_sequence()
c_streams, c_meta = load_streams(C_SEQ, CLOUD_METHODS)
c0, c1 = CLOUD_T_START_MS, CLOUD_T_STOP_MS
c_cut = {m: window(ev, c0, c1) for m, ev in c_streams.items()}
rng = np.random.default_rng(0)


def pts(ev):
    return np.column_stack([ev['x'], ev['y'], ev['t'] * 1000.0])


def nearest_gt(ev, gt):
    """Distance from every event to the nearest GT event of the same polarity (inf if GT has none)."""
    d = np.full(ev['t'].size, np.inf)
    for sign in (True, False):
        q, r = (ev['p'] > 0) == sign, (gt['p'] > 0) == sign
        if q.any() and r.any():
            d[q] = cKDTree(pts({k: v[r] for k, v in gt.items()})).query(pts({k: v[q] for k, v in ev.items()}), workers=-1)[0]
    return d


gt = c_cut['ground_truth']
models = [m for m in CLOUD_METHODS if m in c_cut]
DIST = {m: nearest_gt(c_cut[m], gt) for m in models}
summary = []
for m in models:
    ev, d = c_cut[m], DIST[m]
    back = nearest_gt(gt, ev)          # GT -> model: how much of GT is covered
    summary.append({'model': pretty(m), 'events': ev['t'].size, 'GT events': gt['t'].size,
                    f'model events near GT (≤{CLOUD_NEAR_PX:g}) %': 100 * np.mean(d <= CLOUD_NEAR_PX) if d.size else np.nan,
                    f'GT covered (≤{CLOUD_NEAR_PX:g}) %': 100 * np.mean(back <= CLOUD_NEAR_PX) if back.size else np.nan,
                    'mean dist → GT': float(np.mean(d[np.isfinite(d)])) if np.isfinite(d).any() else np.nan})
display(pd.DataFrame(summary).set_index('model').style.format('{:.3g}').set_caption(
    f'{DATASET}/{C_SEQ} · {c0:g}–{c1:g} ms (precision-like / recall-like at {CLOUD_NEAR_PX:g} px)'))


def cap(idx):
    if idx.size <= CLOUD_MAX_POINTS:
        return idx
    return np.sort(rng.choice(idx, CLOUD_MAX_POINTS, replace=False))


def cloud_figure(polarity):
    fig = go.Figure()
    mode_of = []                            # 'gt', 'all', 'near', 'far' for every trace
    sel = (lambda p: p > 0) if polarity == 'ON' else (lambda p: p <= 0) if polarity == 'OFF' else (lambda p: np.ones(p.size, bool))
    g = np.flatnonzero(sel(gt['p']))
    shown = cap(g)
    fig.add_trace(go.Scatter3d(x=gt['x'][shown], y=gt['t'][shown] * 1000, z=gt['y'][shown], mode='markers', name=f'GT ({g.size:,})',
                               marker=dict(size=1.6, color='#9a9a9a', opacity=0.25),
                               hovertemplate='GT<br>x %{x} · y %{z} · t %{y:.3f} ms<extra></extra>'))
    mode_of.append('gt')
    for m in models:
        ev, d = c_cut[m], DIST[m]
        base = sel(ev['p'])
        color = color_of(m, models)
        for mode, keep in (('all', base), ('near', base & (d <= CLOUD_NEAR_PX)), ('far', base & (d > CLOUD_NEAR_PX))):
            idx = cap(np.flatnonzero(keep))
            symbol = np.where(ev['p'][idx] > 0, 'circle', 'diamond') if polarity == 'ON+OFF' else 'circle'
            fig.add_trace(go.Scatter3d(
                x=ev['x'][idx], y=ev['t'][idx] * 1000, z=ev['y'][idx], mode='markers', legendgroup=m,
                name=f'{pretty(m)} · {mode} ({int(keep.sum()):,})', visible=(mode == 'all'),
                marker=dict(size=2.2, color=color, opacity=0.8, symbol=symbol),
                customdata=d[idx], hovertemplate=f'{pretty(m)}<br>x %{{x}} · y %{{z}} · t %{{y:.3f}} ms'
                                                 '<br>dist to GT %{customdata:.2f}<extra></extra>'))
            mode_of.append(mode)
    vis = lambda want: [mo == 'gt' or mo == want for mo in mode_of]
    fig.update_layout(
        template='plotly_white', height=720, margin=dict(l=0, r=0, t=90, b=0),
        title=f'{polarity} events · {DATASET}/{C_SEQ} · {c0:g}–{c1:g} ms · grey = GT'
              + (' · ● ON ◆ OFF' if polarity == 'ON+OFF' else ''),
        legend=dict(itemsizing='constant'),
        scene=dict(xaxis_title='x (px)', yaxis_title='t (ms)', zaxis_title='y (px)',
                   zaxis=dict(autorange='reversed'), aspectmode='manual', aspectratio=dict(x=1.3, y=1.8, z=1.0),
                   camera=dict(eye=dict(x=1.6, y=-1.6, z=0.9))),
        updatemenus=[dict(type='buttons', direction='right', x=0, y=1.08, xanchor='left', buttons=[
            dict(label='All events', method='restyle', args=['visible', vis('all')]),
            dict(label=f'Near GT only (≤ {CLOUD_NEAR_PX:g})', method='restyle', args=['visible', vis('near')]),
            dict(label=f'Far from GT (> {CLOUD_NEAR_PX:g})', method='restyle', args=['visible', vis('far')])])])
    return fig


tag = '+'.join(safe(m) for m in models)
for polarity in ('ON', 'OFF', 'ON+OFF'):
    fig = cloud_figure(polarity)
    show_plotly(fig)
    save_fig(fig, f'cloud3d_{polarity.replace("+", "")}_{safe(DATASET)}_{safe(C_SEQ)}_{tag}_{c0:g}-{c1:g}ms.html')